# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [6]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [19]:
from dotenv import load_dotenv, find_dotenv
import os
from langchain_openai import ChatOpenAI

_ = load_dotenv(find_dotenv())

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = "My_deployment:dXMtY2VudHJhbDEuZ2NwLmNsb3VkLmVzLmlvOjQ0MyQ2NDU1MmE4M2M2ZWI0OTYzYTIzNjYzNDY4YTg5YmY5ZSRiNzc4ZDBiOWE3Mjk0NjdiODg5ODZhNzFhNjVjNDUxNg=="
if ELASTIC_CLOUD_ID:
    ELASTIC_CLOUD_ID = ELASTIC_CLOUD_ID.strip()

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")

OPENAI_API_KEY = os.getenv("OPENAI_TEST_KEY_KdR")

# https://platform.openai.com/api-keys
# Check if keys and cloud ID are loaded and properly formatted
if not all([ELASTIC_CLOUD_ID, ELASTIC_API_KEY, OPENAI_API_KEY]):
    raise ValueError("Please set ELASTIC_CLOUD_ID, ELASTIC_API_KEY, and OPENAI_TEST_KEY_KdR in your environment variables.")
if not all([ELASTIC_CLOUD_ID, ELASTIC_API_KEY, OPENAI_API_KEY]):
    raise ValueError("Please set ELASTIC_CLOUD_ID, ELASTIC_API_KEY, and OPENAI_TEST_KEY_KdR in your environment variables.")

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name= "ipa7",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [11]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [12]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name", "")
    metadata["summary"] = record.get("summary", "")
    metadata["url"] = record.get("url", "")
    metadata["category"] = record.get("category", "")
    metadata["updated_at"] = record.get("updated_at", "")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [21]:
# Clean up metadata to avoid nested dicts/lists and ensure all values are strings
for doc in docs:
    if hasattr(doc, "metadata"):
        keys_to_delete = []
        for key, value in list(doc.metadata.items()):
            # Remove keys with None or empty string values
            if value is None or value == "":
                keys_to_delete.append(key)
            # Convert dicts/lists to string
            elif isinstance(value, (dict, list)):
                doc.metadata[key] = str(value)
            # Ensure all values are strings
            elif not isinstance(value, str):
                doc.metadata[key] = str(value)
        for key in keys_to_delete:
            del doc.metadata[key]

# Use the class method to create and index documents
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="ipa7",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = ChatOpenAI(temperature=0.5, model="gpt-3.5-turbo", api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [22]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information about the sales team at NASA?', '2. How is the sales team structured within NASA?', '3. What roles and responsibilities does the sales team have at NASA?']


---- Answer ----
content="The NASA sales team refers to the North America South America region within the sales organization structure described in the provided context. This region includes the United States, Canada, Mexico, as well as Central and South America. The NASA sales team is overseen by two Area Vice-Presidents: Laura Martinez, who is the Area Vice-President of North America, and Gary Johnson, who is the Area Vice-President of South America. \n\nThe North America South America region is strategically divided to cater to the specific market dynamics and customer needs of the countries within these continents. Each Area Vice-President is responsible for leading their respective sales team, which includes dedicated account managers, sales representatives, and support staff. The NASA sales team's main objective is to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction across the diverse markets present in North 

**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?